# SNC — Text-Queried Sound Removal (AudioSep)

Removes specific sounds from any audio or video file using
[**AudioSep**](https://github.com/Audio-AGI/AudioSep) — a text-queried
**universal** source separation model (*"Separate Anything You
Describe"*).

This is **not** a music-only tool. AudioSep is trained on AudioSet —
2M+ clips spanning 500+ sound classes — so it can isolate and remove
almost any sound you can describe in words:

- **Vehicles:** car engine, car horn, motorcycle, truck, siren, train
- **Nature / weather:** wind, rain, thunder, ocean waves, flowing water
- **Animals:** barking dog, meowing cat, birds chirping, insects
- **Human:** speech, crowd chatter, laughter, footsteps, crying baby
- **Household / machinery:** keyboard typing, vacuum cleaner, drill, fan
- **Alarms:** smoke alarm, alarm clock, telephone ringing
- **Music:** drums, electric guitar, piano, singing voice

You describe the sounds to remove in plain English; the model isolates
each one and we subtract it from the mixture.

**This notebook is fully self-contained — it does not use the project's
ESC-50 dataset or any local data. It only needs the file you upload.**

## Example use cases

| Input | Sounds to remove | Result |
|-------|------------------|--------|
| Street video | `['car engine', 'car horn', 'wind noise']` | Video with traffic noise removed |
| Nature recording | `['barking dog', 'people talking']` | Just the birds and ambience |
| Podcast clip | `['background music']` | Speech-only audio |
| Music track | `['drums', 'electric guitar']` | Track without drums & guitar |

## Before running

**Runtime → Change runtime type → GPU (T4 or better).** AudioSep runs on
CPU but is ~30x slower.

> AudioSep is a research repo. If an install cell fails on a Colab
> dependency change, that's expected — report the error and it can be
> pinned.

## 1. Check the GPU runtime

In [ ]:
import torch
print('PyTorch        :', torch.__version__)
print('CUDA available :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU            :', torch.cuda.get_device_name(0))
else:
    print('WARNING: no GPU — separation will be very slow.')

## 2. Install AudioSep

Clones the AudioSep repo and installs the dependencies Colab does not
already ship. We deliberately do **not** reinstall torch — Colab's
build is GPU-matched.

In [ ]:
import os, subprocess
from pathlib import Path

AUDIOSEP_DIR = Path('/content/AudioSep')
if not AUDIOSEP_DIR.exists():
    subprocess.run(
        ['git', 'clone', 'https://github.com/Audio-AGI/AudioSep.git', str(AUDIOSEP_DIR)],
        check=True,
    )
os.chdir(AUDIOSEP_DIR)
print('AudioSep repo at', os.getcwd())

In [ ]:
# Dependencies not preinstalled on Colab. torch/torchaudio are left alone.
!pip install -q torchlibrosa pytorch-lightning braceexpand webdataset ftfy gdown
print('Dependencies installed.')

## 3. Download model checkpoints

Two checkpoints from the public AudioSep HuggingFace Space (~2 GB
total). The download runs on Colab's network — it does not use your
home bandwidth. `wget -c` resumes if interrupted.

In [ ]:
os.makedirs('checkpoint', exist_ok=True)

CKPTS = {
    'checkpoint/audiosep_base_4M_steps.ckpt':
        'https://huggingface.co/spaces/Audio-AGI/AudioSep/resolve/main/checkpoint/audiosep_base_4M_steps.ckpt',
    'checkpoint/music_speech_audioset_epoch_15_esc_89.98.pt':
        'https://huggingface.co/spaces/Audio-AGI/AudioSep/resolve/main/checkpoint/music_speech_audioset_epoch_15_esc_89.98.pt',
}
for path, url in CKPTS.items():
    if os.path.exists(path) and os.path.getsize(path) > 1_000_000:
        print('Already downloaded:', path)
    else:
        print('Downloading', path, '...')
        subprocess.run(['wget', '-c', '-q', '--show-progress', '-O', path, url], check=True)
print('Checkpoints ready.')

## 4. Load the AudioSep model

In [ ]:
from pipeline import build_audiosep, inference

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = build_audiosep(
    config_yaml='config/audiosep_base.yaml',
    checkpoint_path='checkpoint/audiosep_base_4M_steps.ckpt',
    device=device,
)
print('AudioSep loaded on', device)

## 5. Upload your audio or video file

Supported: `.wav .mp3 .m4a .flac .ogg` (audio) and
`.mp4 .mov .mkv .avi .webm` (video).

Keep clips under ~3 minutes for the first try — long files work but are
processed in 30-second chunks and take proportionally longer.

In [ ]:
from google.colab import files

uploaded = files.upload()
INPUT_FILE = '/content/' + list(uploaded.keys())[0]
with open(INPUT_FILE, 'wb') as f:
    f.write(uploaded[list(uploaded.keys())[0]])

ext = os.path.splitext(INPUT_FILE)[1].lower()
IS_VIDEO = ext in {'.mp4', '.mov', '.mkv', '.avi', '.webm'}
print(f'Uploaded: {INPUT_FILE}  (video={IS_VIDEO})')

## 6. Choose which sounds to remove

Edit the list below. Each entry is a plain-English description of a
sound to remove. AudioSep is text-queried — be specific
(`'electric guitar'` works better than `'guitar'`).

In [ ]:
# --- EDIT THIS LIST ---
# Plain-English descriptions of the sounds to REMOVE. Works for any
# sound type, not just music. Be specific where it helps
# ('electric guitar' beats 'guitar'; 'car horn' beats 'horn').
SOUNDS_TO_REMOVE = ['car engine', 'car horn', 'wind noise']

# More examples — copy any line over the one above:
#   SOUNDS_TO_REMOVE = ['barking dog']
#   SOUNDS_TO_REMOVE = ['people talking', 'crowd noise']
#   SOUNDS_TO_REMOVE = ['rain', 'thunder']
#   SOUNDS_TO_REMOVE = ['vacuum cleaner', 'keyboard typing']
#   SOUNDS_TO_REMOVE = ['background music']
#   SOUNDS_TO_REMOVE = ['drums', 'electric guitar']
# ----------------------
print('Will remove:', SOUNDS_TO_REMOVE)

## 7. Run separation and removal

For each described sound, AudioSep predicts a spectrogram mask on the
mixture (so the isolated sound is phase-aligned). We sum the isolated
sounds and subtract them from the original — the residual is your file
with those sounds removed.

In [ ]:
import numpy as np
import librosa
import soundfile as sf

SR = 32000  # AudioSep operates at 32 kHz

# Extract a 32 kHz mono WAV from the upload (works for audio and video).
mix_wav = '/content/_mixture.wav'
subprocess.run(
    ['ffmpeg', '-y', '-loglevel', 'error', '-i', INPUT_FILE,
     '-ac', '1', '-ar', str(SR), mix_wav],
    check=True,
)
mixture, _ = librosa.load(mix_wav, sr=SR, mono=True)
print(f'Mixture: {len(mixture) / SR:.1f}s')


def separate_query(query: str, chunk_s: int = 30) -> np.ndarray:
    """Isolate `query` from the mixture, processing in chunks for long files."""
    isolated = np.zeros_like(mixture)
    chunk = chunk_s * SR
    for start in range(0, len(mixture), chunk):
        seg = mixture[start:start + chunk]
        sf.write('/content/_seg.wav', seg, SR)
        inference(model, '/content/_seg.wav', query, '/content/_seg_out.wav', device)
        seg_out, _ = librosa.load('/content/_seg_out.wav', sr=SR, mono=True)
        n = min(len(seg_out), len(seg))
        isolated[start:start + n] += seg_out[:n]
    return isolated


removed_sum = np.zeros_like(mixture)
for query in SOUNDS_TO_REMOVE:
    print(f'  isolating: "{query}" ...')
    removed_sum += separate_query(query)

residual = mixture - removed_sum
peak = np.max(np.abs(residual))
if peak > 1.0:
    residual = residual / peak  # prevent clipping

CLEAN_WAV = '/content/cleaned_audio.wav'
sf.write(CLEAN_WAV, residual, SR)
print('Saved cleaned audio:', CLEAN_WAV)

## 8. Build the final file (remux video if needed)

In [ ]:
stem = os.path.splitext(os.path.basename(INPUT_FILE))[0]

if IS_VIDEO:
    OUTPUT_FILE = f'/content/cleaned_{stem}{ext}'
    subprocess.run(
        ['ffmpeg', '-y', '-loglevel', 'error',
         '-i', INPUT_FILE, '-i', CLEAN_WAV,
         '-c:v', 'copy', '-map', '0:v:0', '-map', '1:a:0', '-shortest',
         OUTPUT_FILE],
        check=True,
    )
else:
    OUTPUT_FILE = f'/content/cleaned_{stem}.wav'
    sf.write(OUTPUT_FILE, residual, SR)

print('Final output:', OUTPUT_FILE)

## 9. Preview and download

In [ ]:
from IPython.display import Audio, Video, display

print('Original mixture:')
display(Audio(mixture, rate=SR))
print('After removal:')
display(Audio(residual, rate=SR))

if IS_VIDEO:
    display(Video(OUTPUT_FILE, embed=True))

In [ ]:
files.download(OUTPUT_FILE)